# Lab 3: Conversation Memory & Tool Calling
## CCI Session 3

**Duration:** 15 minutes

### Clinical Scenario
> You are building a clinical assistant for KHCC nurses. The assistant must remember the conversation context (so nurses can ask follow-up questions) and have access to tools that look up patient vitals and check reference ranges.

### Objective
- Build conversation memory using a messages list
- Implement OpenAI tool calling with clinical tools
- Create a multi-turn clinical assistant

---
### Setup

In [ ]:
!pip install -q openai tiktoken

In [ ]:
import json
from openai import OpenAI

# Paste your OpenAI API key here
OPENAI_API_KEY = ""  # <-- paste your key between the quotes

client = OpenAI(api_key=OPENAI_API_KEY)
print("Client ready!")

---
## Cell 1 — The Stateless Problem
First, let's see why memory matters. Make two API calls without memory.

In [ ]:
# === CELL 1: THE STATELESS PROBLEM ===

# Call 1: Ask about a patient
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Patient MRN 18887 has a temperature of 101.5°F. Is this concerning?"}]
)
print("Response 1:", response1.choices[0].message.content)

# Call 2: Follow-up WITHOUT memory
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What should the nurse do about it?"}]
)
print("\nResponse 2:", response2.choices[0].message.content)

# TODO: Notice — the model has NO idea what "it" refers to!

---
## Cell 2 — Building Conversation Memory
Maintain a messages list and send the full history each time.

In [ ]:
# === CELL 2: CONVERSATION MEMORY ===

# TODO: Create a messages list with a system message
# System: "You are a clinical assistant at KHCC. Help nurses with patient care questions."
messages = []

# TODO: Write a function send_message(user_input) that:
# 1. Appends the user message to the messages list
# 2. Calls client.chat.completions.create() with the full messages list
# 3. Appends the assistant response to the messages list
# 4. Returns the response text

def send_message(user_input):
    pass  # TODO

# TODO: Test with a multi-turn conversation:
# Turn 1: "Patient MRN 18887 has a temperature of 101.5°F. Is this concerning?"
# Turn 2: "What should the nurse do about it?"
# Turn 3: "Should we also check their blood pressure?"
# Notice: Turn 2 and 3 now correctly reference previous context!

---
## Cell 3 — Token Budget Management
As conversations grow, the token count increases. Let's track it.

In [ ]:
# === CELL 3: TOKEN BUDGET ===
import tiktoken

# TODO: Write a function count_conversation_tokens(messages) that
# counts the total tokens in the conversation history
# Hint: use tiktoken.encoding_for_model("gpt-4o-mini")
# Then concatenate all message contents and encode

def count_conversation_tokens(messages):
    pass  # TODO

# TODO: After each send_message call, print the token count
# Discuss: what happens when the conversation reaches 128K tokens?

---
## Cell 4 — Clinical Tool Definitions
Define tools the model can call to look up patient data.

In [ ]:
# === CELL 4: CLINICAL TOOLS ===

# Mock patient database (simulating KHCC data)
patient_db = {
    "18887304731609": {
        "vitals": {"temperature": 101.5, "pulse": 95, "bp": "130/85", "spo2": 96, "respiration": 20},
        "ward": "KHCC-4C-HUSSAM AL-HARIRI",
        "last_updated": "2026-01-16T12:32:00"
    },
    "21307935080461": {
        "vitals": {"temperature": 98.6, "pulse": 72, "bp": "120/80", "spo2": 98, "respiration": 16},
        "ward": "KHCC-3B-ONCOLOGY",
        "last_updated": "2026-01-16T14:00:00"
    }
}

# Clinical reference ranges
reference_ranges = {
    "temperature": {"low": 97.8, "high": 99.1, "unit": "°F"},
    "pulse": {"low": 60, "high": 100, "unit": "bpm"},
    "spo2": {"low": 95, "high": 100, "unit": "%"},
    "respiration": {"low": 12, "high": 20, "unit": "breaths/min"}
}

# TODO: Define three Python functions:
# 1. lookup_vitals(mrn: str) -> dict — returns patient vitals from patient_db
#    If MRN not found, return {"error": "Patient not found"}
# 2. check_abnormal_range(vital_type: str, value: float) -> dict
#    Returns {"vital": ..., "value": ..., "status": "normal"/"high"/"low", "reference": ...}
# 3. get_patient_summary(mrn: str) -> dict — returns full patient info

def lookup_vitals(mrn):
    pass  # TODO

def check_abnormal_range(vital_type, value):
    pass  # TODO

def get_patient_summary(mrn):
    pass  # TODO

---
## Cell 5 — OpenAI Tool Definitions
Define the tools in OpenAI's JSON Schema format so the model knows what it can call.

In [ ]:
# === CELL 5: OPENAI TOOL DEFINITIONS ===

# TODO: Define the tools list in OpenAI format
# Each tool needs: type="function", function={name, description, parameters}
# parameters uses JSON Schema format with type, properties, required

tools = [
    # TODO: Define lookup_vitals tool
    #   name: "lookup_vitals"
    #   description: "Look up the latest vitals for a patient by MRN"
    #   parameters: mrn (string, required)

    # TODO: Define check_abnormal_range tool
    #   name: "check_abnormal_range"
    #   description: "Check if a vital sign value is within normal clinical range"
    #   parameters: vital_type (string, enum of valid types), value (number)

    # TODO: Define get_patient_summary tool
    #   name: "get_patient_summary"
    #   description: "Get a full summary of a patient including vitals, ward, and last update time"
    #   parameters: mrn (string, required)
]

print(f"Defined {len(tools)} tools")

---
## Cell 6 — The Complete Tool Calling Loop
Put it all together: user asks -> model calls tools -> you execute -> model answers.

In [ ]:
# === CELL 6: TOOL CALLING LOOP ===

# Map function names to actual functions
available_functions = {
    "lookup_vitals": lookup_vitals,
    "check_abnormal_range": check_abnormal_range,
    "get_patient_summary": get_patient_summary,
}

# Reset messages for tool-calling conversation
messages = [
    {"role": "system", "content": "You are a clinical assistant at KHCC. Help nurses with patient care questions. Use the available tools to look up patient data and check vital sign ranges."}
]

# TODO: Write a function clinical_assistant(user_input) that:
# 1. Adds user message to messages list
# 2. Calls client.chat.completions.create() with model, messages, and tools=tools
# 3. Gets the response message: response.choices[0].message
# 4. Appends the assistant message to messages list
# 5. Checks if response_message.tool_calls is not None
# 6. If yes, for each tool_call:
#    a. Get the function name: tool_call.function.name
#    b. Parse arguments: json.loads(tool_call.function.arguments)
#    c. Call the actual function: available_functions[name](**args)
#    d. Append tool result: {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
# 7. After processing all tool calls, call the API again with updated messages
# 8. Append and return the final assistant response

def clinical_assistant(user_input):
    pass  # TODO

# TODO: Test with these questions:
# "What are the latest vitals for patient 18887304731609?"
# "Is the patient's temperature normal?"
# "Give me a full summary of patient 21307935080461"

---
## Cell 7 — Multi-Turn with Tools and Memory
Combine memory with tool calling for a full clinical conversation.

In [ ]:
# === CELL 7: MULTI-TURN WITH TOOLS AND MEMORY ===

# TODO: Combine memory (Cell 2) with tool calling (Cell 6)
# The clinical_assistant function already maintains messages history!
# Have a conversation:

# Turn 1: "What are the vitals for patient 18887304731609?"
# print(clinical_assistant("What are the vitals for patient 18887304731609?"))

# Turn 2: "Is anything abnormal?" (should use check_abnormal_range with context from Turn 1)
# print(clinical_assistant("Is anything abnormal?"))

# Turn 3: "Compare with patient 21307935080461"
# print(clinical_assistant("Compare with patient 21307935080461"))

# TODO: Uncomment and run the above lines after completing Cell 6

---
## Stretch Challenge
Add a fourth tool: `recommend_action(vital_type, severity)` that suggests clinical actions based on the vital sign and how far out of range it is.

For example:
- Temperature HIGH + severity "mild" -> "Monitor every 2 hours, administer antipyretic if persistent"
- Temperature HIGH + severity "severe" -> "Immediate blood cultures, notify attending physician"
- SpO2 LOW + severity "mild" -> "Reposition patient, recheck in 15 minutes"
- SpO2 LOW + severity "severe" -> "Initiate supplemental O2, call rapid response team"

In [ ]:
# === STRETCH: RECOMMEND_ACTION TOOL ===
# TODO: Implement the recommend_action function and add it to tools list
pass

---
### KHCC Connection
This tool calling pattern is the foundation for the clinical AI agents you'll build with LangChain in Session 5. The key concepts:

1. **Conversation Memory** — Maintaining a messages list enables multi-turn dialogue
2. **Tool Definitions** — JSON Schema descriptions let the model know what functions are available
3. **Tool Calling Loop** — The pattern of: call API -> execute tools -> call API again is the core agent loop
4. **Clinical Safety** — Tools provide structured, auditable access to patient data

In Session 5, LangChain will abstract this loop into reusable agent architectures.